# Per-Artifact Batch Execute

## What you'll learn

- How the framework decouples overhead amortization from compute dispatch
- The timing difference between sequential and per-artifact parallel execution on Modal
- How to opt out with `per_artifact_dispatch = False` for subprocess-based tools

**Prerequisites:** [First Pipeline](../getting-started/01-first-pipeline.ipynb),
[Batching and Performance](03-batching-and-performance.ipynb),
[Compute Routing](13-compute-routing.ipynb),
[Running on Modal](14-modal-execution.ipynb).
**Estimated time:** 10 minutes
**Modal account required:** Yes (runs compute on Modal containers).

In [ ]:
from __future__ import annotations

import time

from artisan.operations.examples import (
    DataGenerator,
    SequentialSlowTransformer,
    SlowTransformer,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
env = tutorial_setup("batch_execute")

## The decoupled boundary

`artifacts_per_unit` controls how many artifacts share setup overhead
(Delta scans, sandbox creation, staging writes). Within each unit,
**execute is dispatched per-artifact**. On Modal, this means N
containers run in parallel instead of processing all N sequentially
on one container.

```
Unit (N artifacts):

  prep (batched)                     1 Delta scan, 1 sandbox
    ├─ instantiate_inputs()          All N IDs in one query
    ├─ materialize_inputs()          N files into shared dir
    └─ operation.preprocess()        1 call, returns N-item lists

  execute (per-artifact)             N parallel containers on Modal
    ├─ artifact 0  ──→  container
    ├─ artifact 1  ──→  container
    └─ ...         ──→  container

  post (batched)                     1 postprocess, 5 parquet files
    ├─ reassemble results
    ├─ operation.postprocess()       1 call, same data shapes
    └─ record_execution_success()    1 set of staging files
```

| Concern | Parameter | What it controls |
|---------|-----------|------------------|
| Overhead amortization | `artifacts_per_unit` | Delta scans, sandbox, staging |
| Compute parallelism | Per-artifact dispatch | Containers per unit on Modal |

## Seeing the difference: warm containers vs sequential

To make the parallelism visible, we run `SlowTransformer` on Modal
with 8 artifacts. Per-artifact dispatch sends each artifact to its
own container — they execute concurrently within each unit. The shared
router keeps containers warm across units, so only the first unit
pays for cold starts.

Compare this with `SequentialSlowTransformer`, which sets
`per_artifact_dispatch = False` so all artifacts process on a single
container in sequence.

### Generate source data

In [ ]:
pipeline = PipelineManager.create(
    name="batch_execute_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
)
print("Generated 8 source datasets")

### Per-artifact dispatch with warm containers

`SlowTransformer` sleeps 10 seconds per artifact. With
`artifacts_per_unit=4` and 8 artifacts, the step creates 2 units.
`max_workers=1` forces them to run sequentially through the same
shared router. Within each unit, 4 artifacts fan out in parallel
via `spawn_map(4)`.

The first unit cold-starts 4 containers. The second unit hits the
same warm containers — no cold start penalty. `min_containers=4`
tells Modal to keep those containers alive between calls.

In [ ]:
t0 = time.perf_counter()
pipeline.run(
    operation=SlowTransformer,
    name="parallel",
    inputs={"dataset": output("generate", "datasets")},
    params={"duration": 10},
    compute={"active": "modal", "modal": {"min_containers": 4}},
    execution={"artifacts_per_unit": 4, "max_workers": 1},
)
parallel_time = time.perf_counter() - t0
print(f"Per-artifact dispatch (parallel): {parallel_time:.1f}s")

### Sequential dispatch — all artifacts on one container

`SequentialSlowTransformer` sets `per_artifact_dispatch = False`. All 8
artifacts go to one Modal container and sleep sequentially. Total wall
time: ~80s + overhead.

In [ ]:
t0 = time.perf_counter()
pipeline.run(
    operation=SequentialSlowTransformer,
    name="sequential",
    inputs={"dataset": output("generate", "datasets")},
    params={"duration": 10},
    compute_provider="modal",
    execution={"artifacts_per_unit": 8},
)
sequential_time = time.perf_counter() - t0
print(f"Sequential dispatch:              {sequential_time:.1f}s")

In [ ]:
summary = pipeline.finalize()

print(f"\n{'─' * 50}")
print(f"Per-artifact (warm):        {parallel_time:6.1f}s  (4 containers, 2 units)")
print(f"Sequential (one container): {sequential_time:6.1f}s  (1 container)")
print(f"Speedup:                    {sequential_time / parallel_time:5.1f}x")
print(f"{'─' * 50}")

inspect_pipeline(env.delta_root)

## When to opt out

Set `per_artifact_dispatch = False` when your operation runs an
external tool via `run_command()` that loads model weights per
subprocess invocation. Batching amortizes the loading cost — 200
artifacts in one subprocess means one model load.

`SequentialDataTransformer` in the example above is a subclass that
sets this flag:

```python
class SequentialDataTransformer(DataTransformer):
    per_artifact_dispatch: ClassVar[bool] = False
```

This is a stopgap for subprocess-based tools. Operations that keep
state in Python memory (not subprocesses) should use the default
`per_artifact_dispatch = True`.

## Performance model

At 10,000 units of 100 artifacts each (1M total):

| Metric | Per-artifact units (1M of 1) | Batched units (10K of 100) |
|--------|------------------------------|----------------------------|
| Delta scans | 2M | 20K |
| Staging parquet files | 5M | 50K |
| Modal API calls | 1M `remote()` | 10K `spawn_map()` |
| GPU parallelism per unit | 1 container | 100 containers |

## Summary

| Concept | What it does |
|---------|-------------|
| `artifacts_per_unit` | Controls overhead amortization (Delta scans, staging) |
| Per-artifact dispatch | Fans out execute() per artifact within each unit |
| `per_artifact_dispatch = False` | Keeps all artifacts in one execute call |

Operations work without changes. The framework splits preprocess
output, dispatches per-artifact, and reassembles results for
postprocess.

## Next steps

- [Batching and Performance](03-batching-and-performance.ipynb) — `artifacts_per_unit` and overhead amortization
- [Compute Routing](13-compute-routing.ipynb) — Switching between local and Modal compute
- [Running on Modal](14-modal-execution.ipynb) — GPU selection, sandbox transport, debugging
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work